# 19 — Financial Model Builder
> Paste in a plain-English business description and get back a 3-year P&L, free cash flow statement, debt coverage ratio, and a VIABLE / NOT VIABLE verdict — with the specific reason.

*Run all cells. Swap the input in the final code cell with your own data.*

In [ ]:
%pip install -q langchain-openai langchain-core pydantic python-dotenv
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI


# --- Data shapes ---

class FinancialAssumptions(BaseModel):
    revenue_y1: float = Field(description="Year 1 revenue in USD.")
    revenue_growth_rate: float = Field(description="Annual revenue growth rate as a decimal (0.15 = 15%).")
    cogs_pct: float = Field(description="Cost of goods sold as a fraction of revenue (0.40 = 40%).")
    opex_y1: float = Field(description="Year 1 operating expenses in USD, excluding COGS.")
    opex_growth_rate: float = Field(description="Annual opex growth rate as a decimal.")
    tax_rate: float = Field(description="Effective tax rate as a decimal. Applied to positive EBITDA only.")
    capex_y1: float = Field(description="Year 1 capital expenditure in USD.")
    depreciation_y1: float = Field(description="Year 1 depreciation in USD.")
    debt_service_annual: float = Field(
        description="Annual debt service in USD. Use 0.0 if no debt."
    )


class YearlyProjection(BaseModel):
    year: int = Field(description="Projection year (1, 2, or 3).")
    revenue: float = Field(description="Total revenue in USD.")
    cogs: float = Field(description="Cost of goods sold in USD.")
    gross_profit: float = Field(description="Revenue minus COGS.")
    opex: float = Field(description="Operating expenses in USD.")
    ebitda: float = Field(description="Earnings before interest, tax, depreciation, and amortisation.")
    tax: float = Field(description="Tax charge in USD. Zero if EBITDA is negative.")
    net_income: float = Field(description="EBITDA minus tax.")
    fcf: float = Field(description="Free cash flow: EBITDA - tax - capex + depreciation.")


class FinancialModel(BaseModel):
    assumptions: FinancialAssumptions = Field(description="Assumptions extracted from the business brief.")
    projections: list[YearlyProjection] = Field(description="3-year P&L and cash flow projections.")
    dscr: float = Field(
        description="Debt service coverage ratio: average EBITDA / annual debt service. 0.0 if no debt."
    )
    is_viable: bool = Field(
        description="True if net income positive by year 3 and FCF positive by year 2."
    )
    viability_notes: str = Field(description="Plain-English summary of viability assessment.")


# --- Pipeline ---

_EXTRACTOR = SystemMessage(
    """You are a financial analyst. Extract financial assumptions from the business brief.

Rules:
- All rates as decimals: 15% -> 0.15, 40% -> 0.40
- All monetary amounts in USD as floats
- If a value is not stated, estimate based on the business type described
- debt_service_annual: use 0.0 if no debt is mentioned"""
)


def _extract(brief: str) -> FinancialAssumptions:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    chain = _EXTRACTOR | llm.with_structured_output(FinancialAssumptions)
    return chain.invoke(brief)


def _project(a: FinancialAssumptions) -> FinancialModel:
    projections: list[YearlyProjection] = []
    capex = a.capex_y1
    dep = a.depreciation_y1

    for yr in range(1, 4):
        revenue = a.revenue_y1 * (1 + a.revenue_growth_rate) ** (yr - 1)
        cogs = revenue * a.cogs_pct
        gross_profit = revenue - cogs
        opex = a.opex_y1 * (1 + a.opex_growth_rate) ** (yr - 1)
        ebitda = gross_profit - opex
        tax = max(0.0, ebitda * a.tax_rate)
        net_income = ebitda - tax
        fcf = ebitda - tax - capex + dep

        projections.append(
            YearlyProjection(
                year=yr,
                revenue=round(revenue, 2),
                cogs=round(cogs, 2),
                gross_profit=round(gross_profit, 2),
                opex=round(opex, 2),
                ebitda=round(ebitda, 2),
                tax=round(tax, 2),
                net_income=round(net_income, 2),
                fcf=round(fcf, 2),
            )
        )
        capex *= 0.6
        dep *= 1.1

    avg_ebitda = sum(p.ebitda for p in projections) / 3
    dscr = round(avg_ebitda / a.debt_service_annual, 2) if a.debt_service_annual > 0 else 0.0

    y2, y3 = projections[1], projections[2]
    is_viable = y3.net_income > 0 and y2.fcf > 0

    concerns = []
    if y3.net_income <= 0:
        concerns.append("net income negative in year 3")
    if y2.fcf <= 0:
        concerns.append("FCF negative in year 2")
    if a.debt_service_annual > 0 and dscr < 1.25:
        concerns.append(f"DSCR {dscr:.2f}x below 1.25x minimum")

    return FinancialModel(
        assumptions=a,
        projections=projections,
        dscr=dscr,
        is_viable=is_viable,
        viability_notes="Viable" if is_viable else f"Concerns: {', '.join(concerns)}",
    )


def run(brief: str) -> FinancialModel:
    """Build a 3-year financial model from an unstructured business brief."""
    return _project(_extract(brief))

## The scenario

A founder is pitching a premium fitness studio concept — two locations planned, heavy fit-out costs in year one, and a loan to fund the buildout. The agent reads the brief, extracts every financial variable, and tells you whether the business can cover its debt and turn a profit before year three is out.

In [ ]:
brief = """
We're opening two boutique fitness studios in 2025. Combined year-1 revenue is projected at $950K,
growing 30% annually as membership builds. Instructor wages and class costs run at 45% of revenue.
Fixed operating costs (rent, admin, marketing) are $320K in year 1, increasing 10% per year.
We're spending $480K on fit-out and equipment upfront, depreciating $60K per year.
We've secured a $400K SBA loan with annual debt service of $95K. Corporate tax rate is 26%.
"""

model = run(brief)

verdict = "VIABLE" if model.is_viable else "NOT VIABLE"
print(f"Verdict: {verdict}")
print(f"Notes:   {model.viability_notes}")
if model.dscr > 0:
    print(f"DSCR:    {model.dscr:.2f}x  (lenders typically require 1.25x minimum)")

print(f"\n{'Year':<6} {'Revenue':>12} {'Gross Profit':>14} {'EBITDA':>12} {'Net Income':>12} {'FCF':>12}")
print("-" * 70)
for p in model.projections:
    print(
        f"Y{p.year:<5} ${p.revenue:>11,.0f} ${p.gross_profit:>13,.0f}"
        f" ${p.ebitda:>11,.0f} ${p.net_income:>11,.0f} ${p.fcf:>11,.0f}"
    )

print("\n--- Assumptions read from your brief ---")
a = model.assumptions
print(f"Revenue Y1:       ${a.revenue_y1:>10,.0f}")
print(f"Revenue growth:    {a.revenue_growth_rate * 100:.0f}% per year")
print(f"COGS:              {a.cogs_pct * 100:.0f}% of revenue")
print(f"Opex Y1:          ${a.opex_y1:>10,.0f}")
print(f"Capex Y1:         ${a.capex_y1:>10,.0f}")
print(f"Annual debt svc:  ${a.debt_service_annual:>10,.0f}")

## Use your own data

Replace the `brief` string above with:
- Your projected year-1 revenue and annual growth rate
- Your main cost categories (as a percentage of revenue or a flat dollar amount)
- Any debt or loan repayment obligations (or leave them out if you carry none)

The agent will return a year-by-year P&L, your debt coverage ratio, and a clear verdict on whether the numbers work.